<a href="https://colab.research.google.com/github/nivethithanm/python-systems-full/blob/main/triton/T_01_triton_vector_add.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T-01: Triton Vector Add (Hello World)

**Difficulty**: 🟢 Easy | **Category**: Triton Kernels

Your first Triton kernel. Key concepts:
- `@triton.jit` decorator
- `tl.program_id(0)` — which block am I?
- `tl.arange(0, BLOCK_SIZE)` — thread-level offsets within block
- `tl.load(ptr + offsets, mask=mask)` — safe memory access
- `tl.store(ptr + offsets, value, mask=mask)` — write results

**Install**: `pip install triton`

In [1]:
import torch
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print('Install triton: pip install triton')

In [4]:
if HAS_TRITON:
    @triton.jit
    def add_kernel(
        x_ptr, y_ptr, output_ptr,
        n_elements,
        BLOCK_SIZE: tl.constexpr,
    ):
        """Each program instance processes BLOCK_SIZE elements."""
        # TODO: Get this program's block index
        pid = tl.program_id(0)  # tl.program_id(0)

        # TODO: Compute offsets for this block
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

        # TODO: Create mask for bounds checking
        mask = offsets < n_elements  # offsets < n_elements

        # TODO: Load, add, store
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        tl.store(output_ptr + offsets, x + y, mask=mask)

    def triton_add(x, y):
        output = torch.empty_like(x)
        n = x.numel()
        BLOCK_SIZE = 1024
        grid = ((n + BLOCK_SIZE - 1) // BLOCK_SIZE,)
        add_kernel[grid](x, y, output, n, BLOCK_SIZE=BLOCK_SIZE)
        return output

    # Test
    x = torch.randn(10000, device='cuda')
    y = torch.randn(10000, device='cuda')
    out = triton_add(x, y)
    assert torch.allclose(out, x + y, atol=1e-5)
    print('✅ Triton Vector Add passed!')

✅ Triton Vector Add passed!
